# 强化学习 Reinforcement Learning

<img src="https://raw.githubusercontent.com/LisonEvf/practicalAI-cn/master/images/logo.png" width=150>

强化学习是一种通过与环境交互学习最优策略的机器学习范式，智能体通过试错学习最大化累积奖励。

Reinforcement learning is a machine learning paradigm that learns optimal policies through interaction with the environment, where agents learn by trial and error to maximize cumulative rewards.

<img src="https://raw.githubusercontent.com/LisonEvf/practicalAI-cn/master/images/rl.png" width=500>

# 概述 Overview

* **目标:**  学习最大化累积奖励的最优策略。
* **优点:** 
  * 可处理序列决策问题
  * 无需监督信号
  * 适用于复杂环境
* **缺点:**
  * 样本效率低
  * 训练不稳定
  * 超参数敏感
* **其他:** 
  * 核心概念：状态、动作、奖励、策略
  * 广泛应用：游戏、机器人、控制

# 设置 Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

# 环境定义 Environment Definition

In [ ]:
# 简化的网格世界环境
# Simplified Grid World Environment

class GridWorld:
    def __init__(self, size=5):
        self.size = size
        self.start = (0, 0)
        self.goal = (size-1, size-1)
        self.state = self.start
    
    def reset(self):
        self.state = self.start
        return self.state
    
    def step(self, action):
        # 动作: 0=上, 1=右, 2=下, 3=左
        moves = [(-1, 0), (0, 1), (1, 0), (0, -1)]
        new_state = tuple(np.array(self.state) + np.array(moves[action]))
        
        # 检查边界
        if 0 <= new_state[0] < self.size and 0 <= new_state[1] < self.size:
            self.state = new_state
        
        # 计算奖励
        if self.state == self.goal:
            reward = 1.0
            done = True
        else:
            reward = -0.01  # 每步都有小幅惩罚，鼓励快速到达
            done = False
        
        return self.state, reward, done
    
    def render(self):
        grid = [['.' for _ in range(self.size)] for _ in range(self.size)]
        grid[self.start[0]][self.start[1]] = 'S'
        grid[self.goal[0]][self.goal[1]] = 'G'
        if self.state != self.goal:
            grid[self.state[0]][self.state[1]] = 'A'
        print('\n'.join([' '.join(row) for row in grid]))
        print()

# Q-learning 算法 Q-Learning Algorithm

In [ ]:
# Q-learning: 无模型、离策略算法
# Q-learning: model-free, off-policy algorithm

class QLearningAgent:
    def __init__(self, n_states, n_actions, learning_rate=0.1, gamma=0.99, epsilon=0.1):
        self.n_states = n_states
        self.n_actions = n_actions
        self.lr = learning_rate
        self.gamma = gamma
        self.epsilon = epsilon
        self.q_table = defaultdict(lambda: np.zeros(n_actions))
    
    def choose_action(self, state):
        # epsilon-greedy
        if np.random.random() < self.epsilon:
            return np.random.randint(self.n_actions)
        else:
            return np.argmax(self.q_table[state])
    
    def update(self, state, action, reward, next_state, done):
        # Q-learning 更新规则
        if done:
            target = reward
        else:
            target = reward + self.gamma * np.max(self.q_table[next_state])
        
        # 更新Q值
        self.q_table[state][action] += self.lr * (target - self.q_table[state][action])
    
    def train(self, env, n_episodes):
        rewards = []
        for episode in range(n_episodes):
            state = env.reset()
            total_reward = 0
            done = False
            
            while not done:
                action = self.choose_action(state)
                next_state, reward, done = env.step(action)
                self.update(state, action, reward, next_state, done)
                state = next_state
                total_reward += reward
            
            rewards.append(total_reward)
            
            if (episode + 1) % 20 == 0:
                avg_reward = np.mean(rewards[-20:])
                print(f"Episode {episode+1}, Avg Reward (last 20): {avg_reward:.3f}")
        
        return rewards

# 训练智能体 Train Agent

In [ ]:
# 创建环境
env = GridWorld(size=5)
n_states = env.size * env.size
n_actions = 4

# 创建智能体
agent = QLearningAgent(
    n_states=n_states,
    n_actions=n_actions,
    learning_rate=0.2,
    gamma=0.99,
    epsilon=0.3
)

# 训练
print("Training Q-Learning Agent...")
rewards = agent.train(env, n_episodes=200)

# 可视化训练过程 Visualize Training

In [ ]:
plt.figure(figsize=(10, 5))
# 移动平均
window = 10
smoothed = np.convolve(rewards, np.ones(window)/window, mode='valid')
plt.plot(smoothed)
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.title('Q-Learning Training Progress')
plt.grid(True)
plt.show()

# 测试训练好的策略 Test Trained Policy

In [ ]:
# 测试智能体
def test_agent(agent, env, n_episodes=5):
    for episode in range(n_episodes):
        state = env.reset()
        done = False
        
        print(f"Episode {episode + 1}:")
        env.render()
        
        steps = 0
        while not done and steps < 50:
            action = np.argmax(agent.q_table[state])
            next_state, reward, done = env.step(action)
            state = next_state
            steps += 1
            
            if done:
                env.render()
                print(f"Reached goal in {steps} steps!")
                break
        
        if not done:
            print(f"Failed to reach goal in {steps} steps")
        print()

agent.epsilon = 0  # 测试时不探索
test_agent(agent, env)

# 可视化Q值 Visualize Q-Values

In [ ]:
# 可视化每个状态的最大Q值
q_values = np.zeros((env.size, env.size))
for i in range(env.size):
    for j in range(env.size):
        state = (i, j)
        q_values[i, j] = np.max(agent.q_table[state])

plt.figure(figsize=(8, 6))
plt.imshow(q_values, cmap='hot', origin='lower')
plt.colorbar(label='Max Q-Value')
plt.title('Q-Value Heatmap (value of each state)')
plt.xlabel('X')
plt.ylabel('Y')

# 标记起点和终点
plt.scatter([0], [0], color='blue', s=100, marker='s', label='Start')
plt.scatter([4], [4], color='green', s=100, marker='*', label='Goal')
plt.legend()
plt.show()

# TODO

- 深度Q网络 DQN (Deep Q-Network)
- 策略梯度 Policy Gradient
- Actor-Critic 算法
- 深度强化学习 Deep RL (AlphaGo, Atari)